# InfraRiskAI EDA 02 - Portfolio Construction

Builds a project-finance portfolio from the local World Bank PPI file and joins available macro/CDS signals where country codes are present.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW = ROOT / 'data' / 'raw'
ppi = pd.read_csv(RAW / 'ppi' / 'ppi_projects.csv', low_memory=False)
wdi = pd.read_csv(RAW / 'worldbank' / 'wdi_macro.csv', low_memory=False)
cds = pd.read_csv(RAW / 'worldbank' / 'cds_spreads.csv', low_memory=False)
ppi.shape, wdi.shape, cds.shape

In [ ]:
display(ppi.columns.to_series().head(30))
sector_col = 'sector' if 'sector' in ppi.columns else ppi.columns[0]
country_col = 'country' if 'country' in ppi.columns else ppi.columns[1]
value_candidates = [c for c in ppi.columns if any(k in c.lower() for k in ['investment', 'capex', 'debt', 'value'])]
sector_col, country_col, value_candidates[:8]

In [ ]:
value_col = value_candidates[0] if value_candidates else None
portfolio = ppi[[country_col, sector_col] + ([value_col] if value_col else [])].copy()
portfolio = portfolio.rename(columns={country_col: 'country', sector_col: 'sector', value_col: 'project_value'})
if 'project_value' in portfolio.columns:
    portfolio['project_value'] = pd.to_numeric(portfolio['project_value'], errors='coerce')
    portfolio = portfolio.sort_values('project_value', ascending=False)
display(portfolio.head(25))
display(portfolio.groupby('sector').size().sort_values(ascending=False).head(10).to_frame('project_count'))

In [ ]:
sector_counts = portfolio['sector'].value_counts().head(10)
ax = sector_counts.plot(kind='bar', figsize=(10, 4), title='Top PPI Sectors by Project Count')
ax.set_xlabel('Sector')
ax.set_ylabel('Projects')
plt.tight_layout()